In [ ]:
import os

def get_prompt_corpus(directory_path: str = "../../dataset/prompts_code_understanding") -> list[str]:
    def read_prompt(path : str):
        with open(path, "r") as f:
            return f.read()

    paths = [os.path.join(directory_path, item) for item in os.listdir(directory_path)]

    return [read_prompt(path) for path in paths]

corpus = get_prompt_corpus()
corpus = list(set(corpus))

43

In [4]:
from pydantic import BaseModel

class Node(BaseModel):
    name: str
    child_nodes: list["Node"]

    def get_formatted_taxonomy(self, level: int = 0) -> str:
        # Generate formatted tree with the indentation specifying how deep the node is
        lines = [f"{'   ' * level}- {self.name}"]
        for child in self.child_nodes:
            formatted_subtree = child.get_formatted_taxonomy(level + 1)
            lines.append(formatted_subtree)
        return "\n".join(lines)

    def get_formatted_leaf_nodes(self, path: str = "") -> list[str]:
        # Generate a list of node traversals to each leaf nodes e.g. ["node1.node2.leaf_node1", ...]
        current_path = f"{path}.{self.name}" if path else self.name

        if not self.child_nodes:
            return [current_path]

        leaf_nodes = []
        for child in self.child_nodes:
            leaf_nodes += child.get_formatted_leaf_nodes(current_path)
        return leaf_nodes

def test_single_node():
    n = Node(name="root", child_nodes=[])
    assert n.get_formatted_taxonomy() == "- root"
    assert n.get_formatted_leaf_nodes() == ["root"]

# Test 2: One branch
def test_one_branch():
    n = Node(
        name="root",
        child_nodes=[
            Node(name="child1", child_nodes=[
                Node(name="leaf1", child_nodes=[])
            ])
        ]
    )
    assert n.get_formatted_taxonomy() == "- root\n   - child1\n      - leaf1"
    assert n.get_formatted_leaf_nodes() == ["root.child1.leaf1"]

# Test 3: Multiple branches and depths
def test_multi_branch():
    n = Node(
        name="root",
        child_nodes=[
            Node(name="A", child_nodes=[
                Node(name="A1", child_nodes=[]),
                Node(name="A2", child_nodes=[]),
            ]),
            Node(name="B", child_nodes=[
                Node(name="B1", child_nodes=[
                    Node(name="B1a", child_nodes=[])
                ])
            ])
        ]
    )
    expected_taxonomy = (
        "- root\n"
        "   - A\n"
        "      - A1\n"
        "      - A2\n"
        "   - B\n"
        "      - B1\n"
        "         - B1a"
    )
    assert n.get_formatted_taxonomy() == expected_taxonomy
    assert sorted(n.get_formatted_leaf_nodes()) == sorted([
        "root.A.A1",
        "root.A.A2",
        "root.B.B1.B1a"
    ])

# Test 4: Empty child_nodes edge case
def test_empty_child_nodes():
    n = Node(name="solo", child_nodes=[])
    assert n.get_formatted_taxonomy() == "- solo"
    assert n.get_formatted_leaf_nodes() == ["solo"]

test_single_node()
test_one_branch()
test_multi_branch()
test_empty_child_nodes()

In [ ]:
# Enrichment
from pydantic import BaseModel
from typing import Optional
from chatgpt import ChatGPTHelper, ChatMessage

class EnrichmentOutput(BaseModel):
    should_split: bool
    reasoning: str
    child_types: list[str] = []

def get_enrichment_system_prompt() -> str:
    return """You are an LLM agent tasked with refining a taxonomy of prompt segment types.

**You will receive:**
- A batch of prompt segments, all labeled with the same segment type.
- The current schema/taxonomy of segment types (with definitions if available).

**Your goal:**
- Decide whether the current segment type should be subdivided into more specific child types.
- Provide clear justification and examples for any proposed split.

**Instructions:**
- Carefully analyze the batch of prompt segments and the definition of their type.
- If the batch contains clear subcategories, propose a split with names, definitions, and examples for each.
- If no split is needed, justify why, and suggest what evidence would trigger a split in the future.

**Output your response in the following JSON format:**

{
  "should_split": "<string: True if you decide to subdivide current segment type else false>",
  "reasoning": "<string: explain your chain-of-thought reasoning>",
  "child_types": [
    {
      "name": "<string: name of child type>",
      "definition": "<string: clear definition>",
      "examples": [
        "<string: example segment 1>",
        "<string: example segment 2>"
      ]
    }
    // Only include if decision is to split else just give an empty list
  ],
}
"""


async def enrichment(current_node: Node, example_node_segments: list[str], root: Node) -> Optional[list[Node]]:
    formatted_input_prompt = f"""Batch: {example_node_segments}\nCurrent segment type: {current_node.name}\nCurrent Schema: {root.get_formatted_taxonomy()}"""
    chat_messages = [
        ChatMessage(role="system", content=get_enrichment_system_prompt()),
        ChatMessage(role="user", content=formatted_input_prompt)
    ]

    chatgpt_helper = ChatGPTHelper()
    response_content = await chatgpt_helper.get_chat_response_content(chat_messages)
    enrichment_output = chatgpt_helper.extract_json_from_response(response_content)

    return EnrichmentOutput(**enrichment_output)


In [7]:
# Segmentation
from collections import defaultdict
from ..backend.chatgpt import ChatGPTHelper, ChatMessage

def get_segmentation_system_prompt(child_nodes: list[str]) -> str:
    return f"""You are a text preprocessing and parsing expert. Your job is to take a given prompt, break it down into meaningful sentences, and classify them into one or more of the following categories:
    - {"\n- ".join(child_nodes)}
    - Miscellaneous: If sentence does not fit into the section types above, place it under Miscellaneous. This could include extra context, notes, or unrelated instructions.

    Note on Python f-strings: Some prompts may contain Python f-strings where curly brackets `{{}}` are used to represent expressions. When you encounter curly brackets in the prompt, treat them as part of a Python expression.

    # Instructions:
    Your task is as follows:
    1. Break the input prompt into complete and meaningful sentences. Ensure each sentence is clear, distinct, and properly punctuated.
        - Non-significant sentences: Ignore non-significant words or segments e.g. ("and", "or") if they are not relevant to the agent's behavior.
    2. Analyze each sentence and classify them into one or more section types. 
        - Prioritize Context: Choose categories that best fits the overall context. For example, if a sentence mentions a feature of the input but is mostly focused on how to handle the input during the task, it belongs under "Task Instructions and Constraints."
        - If you are still unsure whether a sentence belongs to section type A or B, include both.
    3. Return the result as a JSON object in the format below:
    ```json
    [
        {{
            "sentence": <sentence 1>,
            "categories": [<category X>, // add more categories as necessary],
        }},
        {{
            "sentence": <sentence 2>,
            "categories": [<category Y>, // add more categories as necessary],
        }},
        // Add more sentences as necessary
    ]
    ```
    Ensure that all special characters in the keys and values of the generated JSON, such as quotation marks ("), backslashes (\), and other special symbols, are properly escaped.
    """

def collate_sections_from_classified_sentences(sentences_and_segment_types : list[dict]):
    sections = defaultdict(list)
    current_sections = {}

    for sentence_and_segment_type in sentences_and_segment_types:
        sentence, section_types = sentence_and_segment_type["sentence"], sentence_and_segment_type["categories"]
        for section_type in section_types:
            if section_type in current_sections:
                current_sections[section_type] += "\n" + sentence
            else:
                current_sections[section_type] = sentence
        
        # If this sentence is the start of a new section, eject the current sections and delete them from the current_sections dict
        stored_segment_types = list(current_sections.keys())
        for stored_segment_type in stored_segment_types:
            if stored_segment_type not in section_types:
                sections[stored_segment_type].append(current_sections[stored_segment_type])
                del current_sections[stored_segment_type]

    for stored_segment_type, stored_section in current_sections.items():
        sections[stored_segment_type].append(stored_section)
    return sections

async def segmentation(prompt_segment: str, child_nodes: list[str]) -> dict[str, str]:
    chat_messages = [
        ChatMessage(role="system", content=get_segmentation_system_prompt(child_nodes)),
        ChatMessage(role="user", content=f"Input prompt segment:\n{prompt_segment}\nOutput JSON:\n"),
    ]

    chatgpt_helper = ChatGPTHelper()
    response_content = await chatgpt_helper.get_chat_response_content(chat_messages)
    print("content", response_content)
    sentences_and_segment_types = chatgpt_helper.extract_json_from_response(response_content)
    print('Sentence Segments: ', sentences_and_segment_types)
    sections = collate_sections_from_classified_sentences(sentences_and_segment_types)
    return sections

res = await segmentation(corpus[0], ["Role Assignment", "Input Description", "Task", "Examples"])

<>:34: SyntaxWarning: invalid escape sequence '\)'
/var/folders/_x/y4tv1ly96bg0z32tmz9r0bmh0000gn/T/ipykernel_47380/2009832076.py:34: SyntaxWarning: invalid escape sequence '\)'
  """
/var/folders/_x/y4tv1ly96bg0z32tmz9r0bmh0000gn/T/ipykernel_47380/2009832076.py:34: SyntaxWarning: invalid escape sequence '\)'
  """


ImportError: attempted relative import with no known parent package

In [ ]:
import json
import os
from openai import OpenAI
from typing import List, Dict, Any, Optional

class PromptTaxonomyBuilder:
    def __init__(self, corpus_directory_path: str, max_depth: int = 3) -> None:
        """
        Initialize the prompt taxonomy builder.
        
        Args:
            corpus_directory_path: Path to the corpus of prompts
            max_depth: Maximum depth of the taxonomy tree
            api_key: API key for the LLM service
        """
        self.max_depth = max_depth
        def read_prompt(path : str):
            with open(path, "r") as f:
                return f.read()

        paths = [os.path.join(corpus_directory_path, item) for item in os.listdir(corpus_directory_path)]
        self.prompts = [read_prompt(path) for path in paths][:2]
        
        self.openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

        # self.taxonomy = {"root": {"description": "Root level of the taxonomy", "children": {}}}
        # self.llm_determined_leaf_nodes = []  # Track nodes that LLM decided should be leaves
        
    
    def build_taxonomy(self) -> Dict:
        """Build the prompt taxonomy based on the corpus."""
        print(f"Starting taxonomy building with {len(self.prompts)} prompts, max depth: {self.max_depth}")
        
        # Start with the entire prompts as root segments
        segments = [{"text": prompt, "node": "root"} for prompt in self.prompts]
        
        for depth in range(1, self.max_depth + 1):
            print(f"\nProcessing depth {depth}...")
            
            # Process only leaf nodes from previous iteration
            leaf_nodes = self._get_leaf_nodes()
            print(f"Found {len(leaf_nodes)} leaf nodes: {leaf_nodes}")
            
            new_segments = []
            llm_decided = 0  # Count nodes where LLM decided to stop further subdivision
            
            for node in leaf_nodes:
                # Skip nodes that LLM already determined should be leaves
                if node in self.llm_determined_leaf_nodes:
                    continue
                    
                # Get segments belonging to this node
                node_segments = [s for s in segments if s["node"] == node]
                
                if not node_segments:
                    continue
                
                print(f"Processing {len(node_segments)} segments for node: {node}")
                
                # Group segments into mini-batches for processing
                batches = self._create_batches(node_segments)
                
                node_enriched = False
                for batch_idx, batch in enumerate(batches):
                    print(f"Processing batch {batch_idx+1}/{len(batches)} for node {node}")
                    
                    # Segmentation step
                    batch_segments = self._perform_segmentation(batch, node, depth)
                    
                    # Taxonomy enrichment step
                    child_nodes = self._enrich_taxonomy(batch_segments, node)
                    
                    if child_nodes:  # If the LLM decided to subdivide
                        # Add new child nodes to taxonomy
                        self._update_taxonomy(node, child_nodes)
                        new_segments.extend(batch_segments)
                        node_enriched = True
                    else:
                        # LLM decided this node should be a leaf
                        if node not in self.llm_determined_leaf_nodes:
                            self.llm_determined_leaf_nodes.append(node)
                            llm_decided += 1
                
                # If we didn't enrich this node from any batch, it's a LLM-determined leaf
                if not node_enriched and node not in self.llm_determined_leaf_nodes:
                    self.llm_determined_leaf_nodes.append(node)
                    llm_decided += 1
            
            print(f"LLM decided to stop further subdivision on {llm_decided} nodes at depth {depth}")
            
            # If no new segments were found, we've reached maximum granularity
            if not new_segments:
                print(f"No new segments identified at depth {depth}. Stopping.")
                break
                
            segments = new_segments
            
            # Save intermediate results
            self._save_taxonomy(f"taxonomy_depth_{depth}.json")
        
        print("\nTaxonomy building complete.")
        print(f"LLM-determined leaf nodes: {self.llm_determined_leaf_nodes}")
        return self.taxonomy
    
    def _get_leaf_nodes(self) -> List[str]:
        """Get all leaf nodes in the current taxonomy."""
        leaf_nodes = []
        
        def traverse(node_path, node_data):
            if not node_data.get("children") or not node_data["children"]:
                leaf_nodes.append(node_path)
            else:
                for child_name, child_data in node_data["children"].items():
                    new_path = f"{node_path}.{child_name}" if node_path != "root" else child_name
                    traverse(new_path, child_data)
        
        traverse("root", self.taxonomy["root"])
        return leaf_nodes
    
    def _create_batches(self, segments: List[Dict], batch_size: int = 5) -> List[List[Dict]]:
        """Group segments into mini-batches for processing."""
        return [segments[i:i + batch_size] for i in range(0, len(segments), batch_size)]
    
    def _perform_segmentation(self, batch: List[Dict], parent_node: str, depth: int) -> List[Dict]:
        """
        Segment each prompt in the batch based on the current taxonomy.
        Returns new segments with their assigned node types.
        """
        segmented_prompts = []
        
        # Prepare the current taxonomy context for the LLM
        taxonomy_context = self._format_taxonomy_for_llm()
        
        for segment in batch:
            # Call LLM for segmentation
            new_segments = self._segment_with_llm(segment["text"], parent_node, taxonomy_context, depth)
            segmented_prompts.extend(new_segments)
            
        return segmented_prompts
    
    def _segment_with_llm(self, text: str, parent_node: str, taxonomy_context: str, depth: int) -> List[Dict]:
        """Use LLM to segment the prompt based on the current taxonomy."""
    
    def _update_taxonomy(self, parent_node: str, child_nodes: Dict[str, Dict]):
        """Update the taxonomy with new child nodes."""
        # Navigate to the parent node
        node_path = parent_node.split('.')
        current = self.taxonomy
        
        for part in node_path:
            if part == "root":
                current = current["root"]
            else:
                current = current["children"][part]
        
        # Add or update child nodes
        if "children" not in current:
            current["children"] = {}
            
        for node_name, node_data in child_nodes.items():
            # Check for overlap with existing nodes and merge if necessary
            # Implementation of node merging logic
            current["children"][node_name] = node_data
            
    def _format_taxonomy_for_llm(self) -> str:
        """Format the current taxonomy for inclusion in LLM prompts."""
        # Convert taxonomy to a readable format for the LLM
        return json.dumps(self.taxonomy, indent=2)